In [1]:
import pandas as pd
import numpy as np
import sklearn as sk
import matplotlib.pyplot as plt
import seaborn as sns
import streamlit as st
import joblib


In [2]:
df = pd.read_csv(r"preprocessed_data_outliers_not_removed.csv")

In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_features = ["area", "bhk", "bathroom", "age"]
categorical_features = ["status", "location", "builder"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

# Log-transform target
y = df["price"]
X = df.drop("price", axis=1)

In [8]:
# MODEL BUILDING
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Lasso
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", Lasso())
])

param_grid = {
    'regressor__alpha': np.logspace(-3, 0, 10)
}

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='neg_mean_squared_error'
)

grid_search.fit(X_train, y_train)

model = grid_search.best_estimator_




In [9]:
from sklearn.metrics import mean_squared_error, r2_score

y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("RMSE:", rmse)

train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

print("Train R²:", r2_score(y_train, train_pred))
print("Test  R²:", r2_score(y_test, test_pred))

RMSE: 26.660803872184303
Train R²: 0.9448356975965606
Test  R²: 0.9466433527236622


In [10]:
new_house = pd.DataFrame({
    "area": [1200],
    "bhk": [2],
    "bathroom": [2],
    "age": [5],
    "status": ["ready"],
    "location": ["Chennai"],
    "builder": ["ABC"]
})

prediction = model.predict(new_house)

price_lakh = prediction[0]
if price_lakh < 1:
    print(f"Predicted Price: ₹{price_lakh * 100000:,.0f}")
elif price_lakh < 100:
    print(f"Predicted Price: ₹{price_lakh:,.2f} lakh")
elif price_lakh < 100000:
    print(f"Predicted Price: ₹{price_lakh / 100:,.2f} crore")
else:
    print(f"Predicted Price: ₹{price_lakh / 100000:,.2f} thousand crore")

Predicted Price: ₹91.97 lakh


In [11]:
joblib.dump(model, "house_price_Lassomodel.pkl")

['house_price_Lassomodel.pkl']